In [11]:
%reload_ext autoreload
%autoreload 2
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
from fun import *
import backtrader as bt

plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号
start_date = dt.date(2025,1,1)
end_date = dt.datetime.today()

# 获取指定日期的日线数据
stock_data = read_day_data(start_date=start_date,end_date=end_date,file_path='ts_stock_all_data',stock_list=['SHSE.600000'])
stock_data = stock_data.drop_nulls(subset=['open','close','pre_close','limit_up','limit_down'])
market_value = read_day_data(start_date=start_date,end_date=end_date,file_path='ts_daily_basic')
market_value = market_value.with_columns([
   ( pl.col('free_share')*pl.col('close')/1e4).alias('free_float_mv')
])
stock_data = stock_data.join(market_value.select(['code','trading_date','free_float_mv']),on=['code','trading_date'],how='left')
#stock_data.schema
# 去掉没用的列
stock_data = stock_data.drop(['change','total_share','attack','activity','pe','float_share','buying','selling','swing','strength','avg_turnover'])

#  将数据处理成backtrader能用的格式(stock_data为pl.DataFrame格式)
def prepare_bt_data(df:pl.DataFrame):
    from mapping import convert_code_format
    df = df.sort(['code','trading_date'])
    df = df.with_columns([
        pl.col('trading_date').cast(pl.Date).alias('date')
    ])
    df = df.rename({
        'trading_date':'datetime',
        'code':'sec_code'
    })
    df = df.select(['sec_code','datetime','open','high','low','close','volume',])
    # 转换为pandas DataFrame
    df = df.to_pandas()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.set_index(['datetime'])
    df['sec_code'] = convert_code_format(df['sec_code'],format='pure')
    df['openinterest'] = 0
    return df

bt_data = prepare_bt_data(stock_data)

In [ ]:
# 实例化大脑
cerebro = bt.Cerebro()
cerebro.broker.setcash(1000000)
cerebro.broker.setcommission(commission=0.001)
cerebro.broker.set_slippage_perc(0.003)

# 按照股票代码,将数据添加到大脑中
for sec_code,group in bt_data.groupby('sec_code'):
    # 去掉sec_code列
    group = group.drop(columns=['sec_code'])
    data = bt.feeds.PandasData(dataname=group,fromdate=start_date,todate=end_date)
    cerebro.adddata(data,name=sec_code)
    print(f"{sec_code} Done !") 

class Mystrategy(bt.Strategy):
    def log(self, txt, dt=None):
        ''' Logging function for this strategy'''
        dt = dt or self.datas[0].datetime.date(0)
        print(f'{dt.isoformat()}, {txt}')
    
    def __init__(self):
        self.order = None  

# 测试类
class TestStrategy(bt.Strategy):
    def log(self, txt, dt=None):
        dt = dt or self.datas[0].datetime.date(0)
        print('%s, %s' % (dt.isoformat(), txt))

    def __init__(self):
        self.dataclose = self.datas[0].close
        self.order = None
        self.buyprice=None
        self.buycomm=None

    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            return

        if order.status in [order.Completed]:
            if order.isbuy():
                self.log(
                    '买入执行，价格：%.2f，成本：%.2f，佣金 %.2f' %
                    (order.executed.price,
                     order.executed.value,
                     order.executed.comm))

                self.buyprice = order.executed.price
                self.buycomm = order.executed.comm
            else:
                self.log('卖出执行，价格：%.2f，成本：%.2f，佣金 %.2f' %
                         (order.executed.price,
                          order.executed.value,
                          order.executed.comm))
            self.bar_executed = len(self)

        elif order.status in [order.Canceled, order.Margin, order.Rejected]:
            self.log('订单取消/保证金不足/拒绝')

        self.order = None

    def notify_trade(self, trade):
        if not trade.isclosed:
            return

        self.log('利润记录，毛利润 %.2f，净利润 %.2f' %
                 (trade.pnl, trade.pnlcomm))
        
    def next(self):
        self.log('收盘价，%.2f' % self.dataclose[0])
        if self.order:
            return

        if not self.position:
            if self.dataclose[0] < self.dataclose[-1]:
                if self.dataclose[-1] < self.dataclose[-2]:
                    self.log('创建买入订单，%.2f' % self.dataclose[0])
                    self.order = self.buy()
        else:
            if len(self) >= (self.bar_executed + 5):
                self.log('创建卖出订单，%.2f' % self.dataclose[0])
                self.order = self.sell()      

600000 Done !


In [ ]:
print(f'初始资金: {cerebro.broker.getvalue()}')
cerebro.addstrategy(TestStrategy)
cerebro.run()

print(f'最终资金: {cerebro.broker.getvalue()}')

初始资金: 999996.6863310102
2025-01-02, 收盘价，10.13
2025-01-02, 创建买入订单，10.13
2025-01-02, 收盘价，10.13
2025-01-02, 创建买入订单，10.13
2025-01-03, 买入执行，价格：10.15，成本：10.15，佣金 0.01
2025-01-03, 收盘价，10.06
2025-01-03, 买入执行，价格：10.15，成本：10.15，佣金 0.01
2025-01-03, 收盘价，10.06
2025-01-06, 收盘价，10.15
2025-01-06, 收盘价，10.15
2025-01-07, 收盘价，10.27
2025-01-07, 收盘价，10.27
2025-01-08, 收盘价，10.30
2025-01-08, 收盘价，10.30
2025-01-09, 收盘价，10.19
2025-01-09, 收盘价，10.19
2025-01-10, 收盘价，10.13
2025-01-10, 创建卖出订单，10.13
2025-01-10, 收盘价，10.13
2025-01-10, 创建卖出订单，10.13
2025-01-13, 卖出执行，价格：10.03，成本：10.15，佣金 0.01
2025-01-13, 利润记录，毛利润 -0.12，净利润 -0.14
2025-01-13, 收盘价，10.00
2025-01-13, 创建买入订单，10.00
2025-01-13, 卖出执行，价格：10.03，成本：10.15，佣金 0.01
2025-01-13, 利润记录，毛利润 -0.12，净利润 -0.14
2025-01-13, 收盘价，10.00
2025-01-13, 创建买入订单，10.00
2025-01-14, 买入执行，价格：10.06，成本：10.06，佣金 0.01
2025-01-14, 收盘价，10.16
2025-01-14, 买入执行，价格：10.06，成本：10.06，佣金 0.01
2025-01-14, 收盘价，10.16
2025-01-15, 收盘价，10.21
2025-01-15, 收盘价，10.21
2025-01-16, 收盘价，10.21
2025-01-16, 收盘价，10.21
2025-01-17

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

最终资金: 999993.3726620203
